# U-Net cell image segmentation

This standalone notebook follows the assignment's U-Net architecture: valid 3 x 3 convolutions, 64 starting channels, transpose-convolution upsampling, and cropped skip connections. Run the cells in order. It will create a reproducible split, train, test, save plots and a report, and package results for GitHub. No evaluation output is prefilled.

Use a Colab GPU runtime. The dataset is not included; supply paired images in `data/cells/scans` and `data/cells/labels`.

## 1. Load data

If your trusted zip file on Google Drive contains `data/cells/...`, uncomment the Drive lines and set `ARCHIVE_PATH`. Otherwise upload or copy `data/cells` into the runtime before running this cell.

In [ ]:
from pathlib import Path
from zipfile import ZipFile

ARCHIVE_PATH = None  # Example: Path('/content/drive/MyDrive/data.zip')
DATA_DIR = Path('data/cells')
SPLIT_PATH = Path('split.json')
RUN_DIR = Path('runs/default')
RESULTS_DIR = Path('results')

# from google.colab import drive
# drive.mount('/content/drive')
if ARCHIVE_PATH is not None:
    with ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(Path('.'))  # Use only your own trusted archive.
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Expected scans and labels under {DATA_DIR.resolve()}')
print('Dataset:', DATA_DIR.resolve())

## 2. Preprocessing

Scans and masks are matched by filename stem. The fixed train/validation/test split is 80/10/10. Training images use flips, rotation, zoom, and gamma correction; validation and test images are unchanged apart from resizing and normalization.

In [ ]:
"""Validate paired cell images, create a reproducible split, and load samples."""

import argparse
import json
import random
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps

try:
    from torch.utils.data import Dataset
except ImportError:  # Pair validation can run before PyTorch is installed.
    Dataset = object


IMAGE_EXTENSIONS = {".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff"}


def _files_by_stem(directory):
    if not directory.is_dir():
        raise FileNotFoundError(f"Missing directory: {directory}")
    files = {}
    for path in directory.iterdir():
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            key = path.stem.casefold()
            if key in files:
                raise ValueError(f"Duplicate image stem in {directory}: {path.stem}")
            files[key] = path
    return files


def discover_pairs(data_dir):
    data_dir = Path(data_dir)
    scans = _files_by_stem(data_dir / "scans")
    labels = _files_by_stem(data_dir / "labels")
    if not scans:
        raise ValueError(f"No scan images found in {data_dir / 'scans'}")
    if scans.keys() != labels.keys():
        missing_labels = sorted(scans.keys() - labels.keys())
        missing_scans = sorted(labels.keys() - scans.keys())
        raise ValueError(
            f"Unpaired files: missing labels for {missing_labels}; "
            f"missing scans for {missing_scans}"
        )
    pairs = []
    for key in sorted(scans):
        scan, label = scans[key], labels[key]
        with Image.open(scan) as image, Image.open(label) as mask:
            if image.size != mask.size:
                raise ValueError(f"Image and mask sizes differ: {scan.name}, {label.name}")
            image.verify()
            mask.verify()
        pairs.append({"scan": scan.name, "label": label.name})
    return pairs


def make_split(pairs, train_fraction=0.8, val_fraction=0.1, seed=42):
    if train_fraction <= 0 or val_fraction <= 0 or train_fraction + val_fraction >= 1:
        raise ValueError("train and validation fractions must be positive and sum to less than 1")
    if len(pairs) < 3:
        raise ValueError("At least three image/mask pairs are needed")
    shuffled = list(pairs)
    random.Random(seed).shuffle(shuffled)
    train_count = min(len(shuffled) - 2, max(1, round(len(shuffled) * train_fraction)))
    val_count = min(len(shuffled) - train_count - 1, max(1, round(len(shuffled) * val_fraction)))
    return {
        "seed": seed,
        "train_fraction": train_fraction,
        "val_fraction": val_fraction,
        "train": sorted(shuffled[:train_count], key=lambda pair: pair["scan"]),
        "val": sorted(shuffled[train_count:train_count + val_count], key=lambda pair: pair["scan"]),
        "test": sorted(shuffled[train_count + val_count:], key=lambda pair: pair["scan"]),
    }


def load_split(path, data_dir):
    with open(path, encoding="utf-8") as handle:
        split = json.load(handle)
    current = {(pair["scan"], pair["label"]) for pair in discover_pairs(data_dir)}
    train = {(pair["scan"], pair["label"]) for pair in split["train"]}
    val = {(pair["scan"], pair["label"]) for pair in split["val"]}
    test = {(pair["scan"], pair["label"]) for pair in split["test"]}
    if not train or not val or not test or train & val or train & test or val & test or train | val | test != current:
        raise ValueError("Split must contain every pair exactly once, with nonempty train, val, and test sets")
    if any(len(group) != len(split[name]) for name, group in (("train", train), ("val", val), ("test", test))):
        raise ValueError("Split contains duplicate pairs")
    return split


class Cell_data(Dataset):
    """PyTorch DataLoader compatible dataset; masks contain class IDs 0 or 1."""

    def __init__(self, data_dir, pairs, size=572, augment=False):
        if size < 320:
            raise ValueError("size must be at least 320 to produce a 128x128 or larger mask")
        self.data_dir = Path(data_dir)
        self.pairs = list(pairs)
        self.size = size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        pair = self.pairs[index]
        with Image.open(self.data_dir / "scans" / pair["scan"]) as source:
            image = source.convert("L").resize((self.size, self.size), Image.Resampling.BILINEAR)
        with Image.open(self.data_dir / "labels" / pair["label"]) as source:
            mask = source.convert("L").resize((self.size, self.size), Image.Resampling.NEAREST)

        if self.augment:
            if random.random() < 0.5:
                image, mask = ImageOps.mirror(image), ImageOps.mirror(mask)
            if random.random() < 0.5:
                image, mask = ImageOps.flip(image), ImageOps.flip(mask)
            turns = random.randrange(4)
            if turns:
                angle = 90 * turns
                image, mask = image.rotate(angle), mask.rotate(angle)

            # Zoom in with the same crop on both images; masks stay categorical.
            if random.random() < 0.5:
                crop_size = round(self.size / random.uniform(1.05, 1.25))
                left = random.randrange(self.size - crop_size + 1)
                top = random.randrange(self.size - crop_size + 1)
                box = (left, top, left + crop_size, top + crop_size)
                image = image.crop(box).resize((self.size, self.size), Image.Resampling.BILINEAR)
                mask = mask.crop(box).resize((self.size, self.size), Image.Resampling.NEAREST)

        image_array = np.asarray(image, dtype=np.float32) / 255.0
        if self.augment and random.random() < 0.5:
            image_array = np.power(image_array, random.uniform(0.7, 1.4)).astype(np.float32)
        image_array = image_array[None, :, :]
        mask_array = (np.asarray(mask) > 0).astype(np.int64)
        return image_array, mask_array


CellDataset = Cell_data  # Backward-compatible name for existing scripts.

In [ ]:
pairs = discover_pairs(DATA_DIR)
split = make_split(pairs, train_fraction=0.8, val_fraction=0.1, seed=42)
write_text = json.dumps(split, indent=2) + '\n'
SPLIT_PATH.write_text(write_text, encoding='utf-8')
print('Image counts:', {name: len(split[name]) for name in ('train', 'val', 'test')})

## 3. Assignment U-Net and metrics

A 572 x 572 input produces a 388 x 388 output. Masks are center-cropped to the prediction size for loss and metrics. Raw logits feed cross entropy loss.

In [ ]:
"""Assignment-style U-Net with valid 3x3 convolutions and cropped skip connections."""

import torch
from torch import nn


def center_crop(tensor, height, width):
    """Crop the spatial center of an image, mask, or feature map."""
    source_height, source_width = tensor.shape[-2:]
    if height > source_height or width > source_width:
        raise ValueError(f"Cannot crop {source_height}x{source_width} to {height}x{width}")
    top = (source_height - height) // 2
    left = (source_width - width) // 2
    return tensor[..., top:top + height, left:left + width]


class twoConvBlock(nn.Module):
    """Valid convolution, ReLU, valid convolution, batch norm, ReLU."""

    def __init__(self, input_channel, output_channel):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(input_channel, output_channel, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.Conv2d(output_channel, output_channel, kernel_size=3),
            nn.BatchNorm2d(output_channel),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.layers(x)


class downStep(nn.Module):
    """Contracting path: four conv/pool stages and one bottleneck block."""

    def __init__(self, base_channels=64):
        super().__init__()
        c = base_channels
        self.blocks = nn.ModuleList(
            [twoConvBlock(1, c), twoConvBlock(c, 2*c),
             twoConvBlock(2*c, 4*c), twoConvBlock(4*c, 8*c)]
        )
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = twoConvBlock(8*c, 16*c)

    def forward(self, x):
        skips = []
        for block in self.blocks:
            x = block(x)
            skips.append(x)
            x = self.pool(x)
        return self.bottleneck(x), skips


class upStep(nn.Module):
    """Transpose convolutions followed by cropped skip concatenation."""

    def __init__(self, base_channels=64):
        super().__init__()
        c = base_channels
        self.upsamples = nn.ModuleList(
            [nn.ConvTranspose2d(16*c, 8*c, 2, stride=2),
             nn.ConvTranspose2d(8*c, 4*c, 2, stride=2),
             nn.ConvTranspose2d(4*c, 2*c, 2, stride=2),
             nn.ConvTranspose2d(2*c, c, 2, stride=2)]
        )
        self.blocks = nn.ModuleList(
            [twoConvBlock(16*c, 8*c), twoConvBlock(8*c, 4*c),
             twoConvBlock(4*c, 2*c), twoConvBlock(2*c, c)]
        )

    def forward(self, x, skips):
        for upsample, block, skip in zip(self.upsamples, self.blocks, reversed(skips)):
            x = upsample(x)
            skip = center_crop(skip, *x.shape[-2:])
            x = block(torch.cat((skip, x), dim=1))
        return x


class UNet(nn.Module):
    def __init__(self, base_channels=64):
        super().__init__()
        if base_channels < 1:
            raise ValueError("base_channels must be positive")
        self.down = downStep(base_channels)
        self.up = upStep(base_channels)
        self.classifier = nn.Conv2d(base_channels, 2, kernel_size=1)

    def forward(self, x):
        x, skips = self.down(x)
        return self.classifier(self.up(x, skips))  # Raw logits for CrossEntropyLoss.

In [ ]:
# Quick shape check before the full training run.
with torch.no_grad():
    check_model = UNet(base_channels=2).eval()
    check_output = check_model(torch.zeros(1, 1, 320, 320))
assert tuple(check_output.shape) == (1, 2, 132, 132)
print('Shape check passed: 320 x 320 input -> 132 x 132 mask')

In [ ]:
"""Aggregate segmentation metrics across an entire data loader."""

import torch
from torch import nn



@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")
    loss_sum = pixels = tp = fp = fn = tn = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        labels = center_crop(labels, *logits.shape[-2:])
        loss_sum += criterion(logits, labels).item()
        predictions = logits.argmax(dim=1)
        tp += ((predictions == 1) & (labels == 1)).sum().item()
        fp += ((predictions == 1) & (labels == 0)).sum().item()
        fn += ((predictions == 0) & (labels == 1)).sum().item()
        tn += ((predictions == 0) & (labels == 0)).sum().item()
        pixels += labels.numel()
    if not pixels:
        raise ValueError("Cannot evaluate an empty dataset")
    return {
        "loss": loss_sum / pixels,
        "pixel_accuracy": (tp + tn) / pixels,
        "dice": (2 * tp / (2 * tp + fp + fn)) if (2 * tp + fp + fn) else 1.0,
        "iou": (tp / (tp + fp + fn)) if (tp + fp + fn) else 1.0,
        "samples": len(loader.dataset),
    }

In [ ]:
"""Create the loss plot and portfolio-ready experiment report."""

import json
from pathlib import Path


def save_loss_plot(history, path, test_loss=None):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    epochs = [row["epoch"] for row in history]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs, [row["train_loss"] for row in history], marker="o", label="Train")
    ax.plot(epochs, [row["loss"] for row in history], marker="o", label="Validation")
    if test_loss is not None:
        ax.axhline(test_loss, color="tab:green", linestyle="--",
                   label="Held out test (evaluated once)")
    ax.set(xlabel="Epoch", ylabel="Cross entropy loss", title="Segmentation loss")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def write_report(summary, metrics, preview_names, output_dir):
    output_dir = Path(output_dir)
    if not summary or not metrics:
        raise ValueError("Training summary and test metrics are required for a report")
    previews = "\n".join(
        f"![Scan, ground truth, prediction for {Path(name).stem}]({name})"
        for name in preview_names
    ) or "No previews were requested."
    architecture_note = (
        "No architecture deviation was used in this run."
        if summary["base_channels"] == 64
        else f"The first block used {summary['base_channels']} channels instead of the assignment's 64."
    )
    report = f"""# Cell segmentation results

This report was generated from a completed run of the assignment-style U-Net. The split was fixed before training. Validation selected the checkpoint; the held out test set was evaluated once.

## Configuration

| Setting | Value |
| --- | ---: |
| Input image size | {summary['image_size']} x {summary['image_size']} |
| Output mask size | {summary['output_height']} x {summary['output_width']} |
| Train / validation / test images | {summary['train_samples']} / {summary['val_samples']} / {summary['test_samples']} |
| Epochs | {summary['epochs']} |
| Best validation epoch | {summary['best_epoch']} |
| Batch size | {summary['batch_size']} |
| Learning rate | {summary['learning_rate']} |
| Device | {summary['device']} |
| Training time | {summary['training_seconds']:.1f} seconds |

The network uses unpadded 3 x 3 convolutions, transpose-convolution upsampling, and center-cropped skip connections. {architecture_note} Images are grayscale and normalized to [0, 1]. Training augmentation uses horizontal and vertical flips, 90-degree rotations, zooming, and gamma correction.

## Loss and test metrics

The chart shows training and validation loss at every epoch. The horizontal test line is the single held out test evaluation, not a per-epoch test measurement.

![Training, validation, and held out test loss](loss_curves.png)

| Held out test metric | Value |
| --- | ---: |
| Cross entropy loss | {metrics['loss']:.4f} |
| Pixel accuracy | {metrics['pixel_accuracy']:.4f} |
| Dice | {metrics['dice']:.4f} |
| Intersection over union | {metrics['iou']:.4f} |
| Test images | {metrics['samples']} |

## Test segmentations

Each preview shows the centered scan crop, ground-truth mask, and prediction from left to right.

{previews}

The split, complete epoch history, and raw metrics are saved in `split.json`, `history.json`, and `metrics.json` for reproducibility.
"""
    (output_dir / "README.md").write_text(report, encoding="utf-8")


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")

## 4. Train and select the checkpoint

The test set is untouched here. The highest validation Dice selects the saved model. A batch size of 1 matches the full 64-channel architecture on common Colab GPUs; adjust it only if your runtime has enough memory.

In [ ]:
import time
from torch.utils.data import DataLoader

EPOCHS = 15
BATCH_SIZE = 1
IMAGE_SIZE = 572  # Use at least 320; 572 -> 388 output, 320 -> 132 output.
BASE_CHANNELS = 64
LEARNING_RATE = 1e-3
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_set = Cell_data(DATA_DIR, split['train'], IMAGE_SIZE, augment=True)
val_set = Cell_data(DATA_DIR, split['val'], IMAGE_SIZE, augment=False)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
model = UNet(BASE_CHANNELS).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = RUN_DIR / 'best_model.pt'
history = []
best_dice = -1.0
best_epoch = 0
output_shape = None
start = time.perf_counter()
print('Training on', device)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    train_samples = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        output_shape = logits.shape[-2:]
        labels = center_crop(labels, *output_shape)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * len(images)
        train_samples += len(images)

    validation = evaluate(model, val_loader, device)
    record = {'epoch': epoch, 'train_loss': train_loss_sum / train_samples, **validation}
    history.append(record)
    write_json(RUN_DIR / 'history.json', history)
    if validation['dice'] > best_dice:
        best_dice = validation['dice']
        best_epoch = epoch
        torch.save({
            'model_state': model.state_dict(),
            'image_size': IMAGE_SIZE,
            'base_channels': BASE_CHANNELS,
            'split': split,
            'epoch': epoch,
        }, CHECKPOINT_PATH)
    print(f"Epoch {epoch}/{EPOCHS}: train loss {record['train_loss']:.4f}, "
          f"val loss {validation['loss']:.4f}, Dice {validation['dice']:.4f}, "
          f"IoU {validation['iou']:.4f}")

training_seconds = time.perf_counter() - start
summary = {
    'image_size': IMAGE_SIZE,
    'output_height': output_shape[0],
    'output_width': output_shape[1],
    'base_channels': BASE_CHANNELS,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'seed': SEED,
    'train_samples': len(train_set),
    'val_samples': len(val_set),
    'test_samples': len(split['test']),
    'device': str(device),
    'training_seconds': training_seconds,
    'best_epoch': best_epoch,
    'best_validation_dice': best_dice,
}
write_json(RUN_DIR / 'training_summary.json', summary)
save_loss_plot(history, RUN_DIR / 'loss_curves.png')
print(f'Best checkpoint: {CHECKPOINT_PATH}; validation Dice: {best_dice:.4f}')
print(f'Training time: {training_seconds:.1f} seconds; output mask: {output_shape}')

## 5. Held out testing and assignment report

This cell reloads the best checkpoint, evaluates the test group once, and saves a plot, metrics, every test segmentation preview, and a report to `results/`. The preview columns are scan crop, true mask, and predicted mask.

In [ ]:
import shutil

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
verified_split = load_split(SPLIT_PATH, DATA_DIR)
if checkpoint['split'] != verified_split:
    raise ValueError('The split differs from the trained checkpoint')

test_set = Cell_data(DATA_DIR, verified_split['test'], checkpoint['image_size'], augment=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)
test_model = UNet(checkpoint['base_channels']).to(device)
test_model.load_state_dict(checkpoint['model_state'])
test_metrics = evaluate(test_model, test_loader, device)
test_metrics['checkpoint_epoch'] = checkpoint['epoch']
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
write_json(RESULTS_DIR / 'metrics.json', test_metrics)

preview_names = []
test_model.eval()
with torch.no_grad():
    for index in range(len(test_set)):
        image, label = test_set[index]
        prediction = test_model(torch.from_numpy(image[None]).to(device)).argmax(dim=1)[0].cpu().numpy()
        height, width = prediction.shape
        image_crop = center_crop(image[0], height, width)
        label_crop = center_crop(label, height, width)
        panels = [
            Image.fromarray((image_crop * 255).astype(np.uint8)),
            Image.fromarray((label_crop * 255).astype(np.uint8)),
            Image.fromarray((prediction * 255).astype(np.uint8)),
        ]
        preview = Image.new('L', (width * 3, height))
        for column, panel in enumerate(panels):
            preview.paste(panel, (column * width, 0))
        name = f"{Path(test_set.pairs[index]['scan']).stem}_preview.png"
        preview.save(RESULTS_DIR / name)
        preview_names.append(name)

save_loss_plot(history, RESULTS_DIR / 'loss_curves.png', test_metrics['loss'])
shutil.copyfile(SPLIT_PATH, RESULTS_DIR / 'split.json')
shutil.copyfile(RUN_DIR / 'history.json', RESULTS_DIR / 'history.json')
write_json(RESULTS_DIR / 'training_summary.json', summary)
write_report(summary, test_metrics, preview_names, RESULTS_DIR)
print(json.dumps(test_metrics, indent=2))
print('Report and previews:', RESULTS_DIR.resolve())

## 6. Inspect and download portfolio results

Review the report and previews before publishing. This cell packages `results/` into `portfolio_results.zip` and offers it as a Colab download.

In [ ]:
from IPython.display import display

print((RESULTS_DIR / 'README.md').read_text(encoding='utf-8'))
display(Image.open(RESULTS_DIR / 'loss_curves.png'))
for name in preview_names:
    print(name, ': scan | true mask | prediction')
    display(Image.open(RESULTS_DIR / name))
archive_path = shutil.make_archive('portfolio_results', 'zip', root_dir='.', base_dir='results')
print('Packaged:', archive_path)
try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print('Download the zip manually from the working directory.')